# Fashion-MNIST: Shallow vs Deep Networks

**Course:** SST Neural Network & Intro to Computer Vision (ML III)  
**Assigned Topic:** Section 2 — 17 Shallow vs. Deep Networks  
**Task:** Train MLPs with 2, 4, and 8 hidden layers on Fashion-MNIST (same total parameter count). Plot accuracy for each depth. Show when deeper networks help and when they fail to train.  

**Team:** Group 8  
- Ujjwal Jain (10173)  
- Pratham Onkar (10136)  
- Aditya Kumar Rai (10178)  
- Dhairya Motta (10202)  
- Arman Barbhuiya (10196)  
- Lakshay Jagga (10398)  
- Piyush Kumar Gupta (10332)  
- Iyad Farooq (10116)  

---


In [ ]:
# Colab Setup: Clone the repository to get access to the src/ folder
import sys, os
if 'google.colab' in sys.modules:
    !git clone https://github.com/Ujjwaljain16/fashion-mnist-depth-study.git
    %cd fashion-mnist-depth-study


In [ ]:
# Cell 1: Environment Setup & Google Drive Mount
import sys, os, shutil
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = '/content/drive/MyDrive/FashionMNIST_Project'
    !git clone https://github.com/Ujjwaljain16/fashion-mnist-depth-study.git repo || true
    %cd repo
else:
    WORKSPACE = '.'
    print('Running locally. Outputs will save to the current directory.')


# Fashion-MNIST: Shallow vs Deep Networks
## Depth Study — 2-Layer vs 4-Layer vs 8-Layer MLPs

**Course:** Deep Learning I &nbsp;|&nbsp; **Assignment:** Section 2 — Network Depth Experiments  
**Dataset:** Fashion-MNIST &nbsp;|&nbsp; **Framework:** PyTorch

---

### Research Questions

| ID | Question |
|----|----------|
| **RQ1** | Does increasing depth improve performance under a fixed parameter budget? |
| **RQ2** | How does depth affect gradient flow? |
| **RQ3** | What role does activation choice play? |
| **RQ4** | Can BatchNorm restore trainability? |

---

> **Notebook usage:** Run all cells top-to-bottom. Set `FAST_MODE = True` for a quick
> test run (2 min on CPU) or `FAST_MODE = False` for the full experiment (~8–15 min on GPU).
> `AUTO_RESUME = True` (default) skips any run whose results already exist in the CSV.

In [ ]:
# Cell 1: Dependency Installation
# Run this cell first. On Colab all packages are pre-installed;

import importlib, subprocess, sys

_required = [
    ("torch",       "torch"),
    ("torchvision", "torchvision"),
    ("numpy",       "numpy"),
    ("pandas",      "pandas"),
    ("matplotlib",  "matplotlib"),
    ("seaborn",     "seaborn"),
]

for _import_name, _pkg_name in _required:
    if importlib.util.find_spec(_import_name) is None:
        print(f"Installing {_pkg_name}...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", _pkg_name],
            check=True
        )

print("All packages ready.")

In [ ]:
# Cell 2: Project Root Discovery + Core Imports
# Locates the project root directory (the one containing src/) and adds it
# to sys.path. Works whether the notebook is run from notebooks/ or the root.

import os, sys

_candidates = [
    os.getcwd(),
    os.path.join(os.getcwd(), ".."),
    os.path.dirname(os.getcwd()),
]
PROJECT_ROOT = None
for _p in _candidates:
    if os.path.isdir(os.path.join(_p, "src")):
        PROJECT_ROOT = os.path.abspath(_p)
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot find project root (directory containing src/).\n"
        "Ensure the notebook lives inside fashion-mnist-depth-study/ and "
        "that src/ exists at the project root."
    )

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)   # Relative paths in CONFIG resolve from here
print(f"Project root : {PROJECT_ROOT}")

import platform
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# src/ imports
from src.utils  import (
    CONFIG, CLASS_NAMES, set_seed,
    get_dataloaders, load_results, load_summary,
    compute_aggregate_summary,
)
from src.models import ConfigurableMLP
from src.train  import ExperimentRunner
from src.plots  import (
    plot_val_accuracy_curves,
    plot_val_loss_curves,
    plot_test_accuracy_bar,
    plot_gradient_norm_by_layer,
    plot_gradient_norm_over_epochs,
    plot_activation_heatmap,
    plot_batchnorm_recovery,
    generate_all_figures,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {DEVICE}")

# DataLoader workers: Windows Jupyter requires num_workers=0
NUM_WORKERS = 0  # Forces main-thread data loading (CRITICAL FOR COLAB SPEED)
print(f"num_workers    : {NUM_WORKERS}")

In [ ]:
# Cell 3: Experiment Configuration
FAST_MODE   = False
AUTO_RESUME = True

if FAST_MODE:
    CONFIG['epochs'] = CONFIG['fast_epochs']
    CONFIG['seeds']  = CONFIG['fast_seeds']

# Redirect all outputs to Google Drive (or local workspace)
CONFIG['results_path']    = f"{WORKSPACE}/results/results.csv"
CONFIG['summary_path']    = f"{WORKSPACE}/results/summary.csv"
CONFIG['figures_dir']     = f"{WORKSPACE}/figures"
CONFIG['checkpoints_dir'] = f"{WORKSPACE}/checkpoints"
CONFIG['exports_dir']     = f"{WORKSPACE}/exports"

for _dir in [CONFIG['data_dir'], Path(CONFIG['results_path']).parent,
             CONFIG['figures_dir'], CONFIG['checkpoints_dir'], CONFIG['exports_dir']]:
    Path(_dir).mkdir(parents=True, exist_ok=True)

print(f"\nCONFIG summary:")
print(f"  batch_size  : {CONFIG['batch_size']}")
print(f"  lr          : {CONFIG['lr']}")
print(f"  epochs      : {CONFIG['epochs']}")
print(f"  seeds       : {CONFIG['seeds']}")
print(f"  results  →  {CONFIG['results_path']}")


---
## Section 1: Introduction

### Problem Statement

Depth is a central hyperparameter in neural network design. Deep learning theory suggests that deeper networks can represent more complex functions using fewer parameters than shallow networks (Telgarsky, 2016). However, increasing depth introduces two practical challenges:

1. **Vanishing Gradients**: Gradients shrink exponentially as they propagate backward through many layers, making early layers unable to learn. This is especially severe with sigmoid activations, where σ'(x) ≤ 0.25 per layer → cumulative attenuation of 0.25^8 ≈ 10⁻⁵ after 8 layers.

2. **Width-Depth Trade-off**: Under a fixed parameter budget, deeper networks require narrower layers. Narrower layers may restrict representational capacity at each level.

### This Study

We compare MLPs of depth 2, 4, and 8 on Fashion-MNIST under a fixed parameter budget of ~500K, systematically investigating:

| RQ | Question | Experiment |
|----|----------|------------|
| RQ1 | Does depth improve performance? | Exp 1 (ReLU, all depths) |
| RQ2 | How does depth affect gradient flow? | Exp 2 (gradient norms) |
| RQ3 | What role does activation play? | Exp 2 (ReLU vs Sigmoid) |
| RQ4 | Can BatchNorm restore trainability? | Exp 3 (8L Sigmoid ±BN) |

---
## Section 2: Dataset — Fashion-MNIST

In [ ]:
# Cell 6: Load Dataset
set_seed(42)
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir    = CONFIG["data_dir"],
    batch_size  = CONFIG["batch_size"],
    val_size    = CONFIG["val_size"],
    train_size  = CONFIG["train_size"],
    norm_mean   = CONFIG["norm_mean"],
    norm_std    = CONFIG["norm_std"],
    seed        = 42,
    num_workers = NUM_WORKERS,
)

print("Fashion-MNIST dataset loaded.")
print(f"  Train      : {len(train_loader.dataset):,} samples")
print(f"  Validation : {len(val_loader.dataset):,} samples")
print(f"  Test       : {len(test_loader.dataset):,} samples")
print(f"  Classes    : {len(CLASS_NAMES)} → {CLASS_NAMES}")
print(f"  Input dim  : 28 × 28 = 784 features (after flatten)")
print(f"  Normalisation: mean={CONFIG['norm_mean']}, std={CONFIG['norm_std']}")
print(f"    Pixel range: [0,1] → [{-CONFIG['norm_mean']/CONFIG['norm_std']:.1f}, "
      f"{(1-CONFIG['norm_mean'])/CONFIG['norm_std']:.1f}]")

In [ ]:
# Cell 7: Visualise Dataset Samples
from torchvision import datasets, transforms

raw_ds = datasets.FashionMNIST(
    CONFIG["data_dir"], train=True, download=False,
    transform=transforms.ToTensor()
)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
shown = {i: 0 for i in range(10)}
count = 0

for img, label in raw_ds:
    r = shown[label]
    if r < 2:
        axes[r, label].imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
        axes[r, label].axis("off")
        if r == 0:
            axes[r, label].set_title(CLASS_NAMES[label], fontsize=7.5)
        shown[label] += 1
        count += 1
    if count == 20:
        break

fig.suptitle("Fashion-MNIST: 2 Raw Samples per Class (Before Normalisation)",
             fontsize=12, y=1.03)
plt.tight_layout()
Path(CONFIG["figures_dir"]).mkdir(parents=True, exist_ok=True)
fig.savefig(f"{CONFIG['figures_dir']}/dataset_samples.png", dpi=150, bbox_inches="tight")
plt.show()

print("Note: During training, images are normalised to [-1, 1] (mean=0.5, std=0.5).")

---
## Section 3: Architecture Design

### Equal Parameter Budget (~500K)

To ensure a **fair comparison**, all models are constrained to approximately the same number of trainable parameters (~500K). This is achieved by reducing layer width as depth increases:

| Depth | Width | Parameter Formula |
|-------|-------|------------------|
| 2L    | 413   | 784×413 + 1×413² + 413×10 |
| 4L    | 296   | 784×296 + 3×296² + 296×10 |
| 8L    | 215   | 784×215 + 7×215² + 215×10 |

> **Why this matters:** Without controlling parameters, deeper networks would have more total capacity. Any accuracy difference would be attributable to *size* rather than *depth*.

### Layer Template

```
Input (784)
  → Flatten
  → [ Linear(in, W) → [BatchNorm1d(W)] → Activation ] × depth
  → Linear(W, 10)           ← output logits (no activation)
```

BatchNorm is **off** for Experiments 1 & 2 (to isolate depth/activation effects)  
BatchNorm is **on** only for the 8L Sigmoid model in Experiment 3.

In [ ]:
#  Cell 9: Instantiate Models and Display Architectures
print("Model architectures (ReLU, no BatchNorm):")

for depth in CONFIG["depths"]:
    width = CONFIG["widths"][depth]
    m = ConfigurableMLP(
        input_dim=CONFIG["input_dim"],
        num_classes=CONFIG["num_classes"],
        depth=depth,
        width=width,
        activation="ReLU",
        use_batchnorm=False,
    )
    print(f"\n{'='*55}")
    print(f"  {m}")
    print(f"  Network:")
    for i, layer in enumerate(m.network):
        print(f"    [{i}] {layer}")

In [ ]:
#  Cell 10: Parameter Count Table
rows = []
for depth in CONFIG["depths"]:
    width = CONFIG["widths"][depth]
    m_no_bn = ConfigurableMLP(
        depth=depth, width=width, activation="ReLU", use_batchnorm=False
    )
    rows.append({
        "Model"     : f"{depth}L MLP (ReLU)",
        "Depth"     : depth,
        "Width"     : width,
        "Parameters": m_no_bn.count_parameters(),
        "Diff from 500K": m_no_bn.count_parameters() - 500_000,
    })

# BatchNorm overhead
m_bn  = ConfigurableMLP(depth=8, width=215, activation="Sigmoid", use_batchnorm=True)
m_nbn = ConfigurableMLP(depth=8, width=215, activation="Sigmoid", use_batchnorm=False)
bn_overhead = m_bn.count_parameters() - m_nbn.count_parameters()

param_df = pd.DataFrame(rows)
print("Parameter Count Table:")
print(param_df.to_string(index=False))
print(f"\nBatchNorm overhead (8L Sigmoid, width=215):")
print(f"  +{bn_overhead:,} params  (= 2 × width × depth = 2 × 215 × 8)")
print(f"  Fraction of total: {bn_overhead/m_bn.count_parameters()*100:.2f}%  (negligible)")
print("\n✓ Widths adjusted so parameter count is approximately constant across depths.")

---
## Section 4: Experiment 1 - Depth Comparison (ReLU)

**Research Question (RQ1):** Does increasing depth improve performance under a fixed parameter budget?

**Setup:**
- Models: 2L, 4L, 8L MLP — all ReLU, no BatchNorm  
- Seeds: 42, 123, 7 (or [42] in FAST_MODE)  
- Epochs: 50 (or 10 in FAST_MODE)  
- All models use ~500K parameters  

**Hypothesis:** Deeper models may learn more hierarchical features, but gains may be marginal on Fashion-MNIST - a relatively simple dataset whose discriminative features (texture, silhouette) may be fully captured by a 2-layer representation.

**What to look for in the results:**
- `Fig 1`: Does a deeper model's accuracy curve converge higher or faster?
- `Fig 2`: Does a deeper model's loss converge lower?
- `Fig 3`: Are differences statistically meaningful given ± std error bars?

In [ ]:
# Experiment 1
runner = ExperimentRunner(config=CONFIG, fast_mode=FAST_MODE, auto_resume=AUTO_RESUME, device=DEVICE, num_workers=NUM_WORKERS, verbose=True)
runner.run_experiment_1()

# Plot figures immediately
plot_val_accuracy_curves(CONFIG['results_path'], CONFIG['figures_dir'], activation='ReLU')
plot_val_loss_curves(CONFIG['results_path'], CONFIG['figures_dir'], activation='ReLU')

# Create Backup
backup_path = f"{CONFIG['exports_dir']}/backup_exp1.zip"
shutil.make_archive(backup_path.replace('.zip', ''), 'zip', Path(CONFIG['results_path']).parent)
print(f"\nExperiment 1 backed up to {backup_path}")


In [ ]:
# Cell 14: Figure 2 - Validation Loss vs Epoch
plot_val_loss_curves(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"],
    activation   = "ReLU",
    batchnorm    = 0,
)


### Experiment 1 Analysis

**Interpreting the depth comparison (Figs 1–3):**

**Depth Invariance:** The grid search reveals that increasing depth under strict capacity control (~500k parameters) yields no statistical improvement for ReLU networks. The 2L (88.79%), 4L (88.79%), and 8L (88.72%) ReLU models perform identically within noise margins (std ~0.003). Under a fixed parameter budget, Fashion-MNIST does not appear to require additional depth to achieve near-optimal performance.

**Overfitting Dynamics:** Figure 2 reveals a hidden dynamic: all ReLU models exhibit severe calibration decay (overfitting), with validation loss nearly doubling by epoch 50. Interestingly, the 8L ReLU model's terminal loss (0.55) is significantly lower than the 2L/4L models (~0.60), suggesting that deeper, narrower MLPs provide stronger implicit regularization under Adam.

---
## Section 5: Experiment 2 — Activation Study (Vanishing Gradients)

**Research Questions (RQ2 + RQ3):** How does depth affect gradient flow? What role does activation choice play?

**Setup:**
- Models: 2L, 4L, 8L × {ReLU, Sigmoid} = 6 configurations (no BatchNorm)
- ReLU runs reused from Experiment 1 via AUTO_RESUME (no re-training)

**Theoretical Background:**

Backpropagation computes gradients via the chain rule. Each layer contributes a factor of the activation derivative:

| Activation | Derivative | Consequence |
|-----------|-----------|-------------|
| Sigmoid | σ'(x) = σ(x)(1−σ(x)) ≤ **0.25** | 0.25^8 ≈ **1.5×10⁻⁵** after 8 layers |
| ReLU | f'(x) = **1** (x>0), **0** (x<0) | Gradients pass unchanged through active neurons |

**Primary Figure: Fig 4A** — Gradient Norm vs Layer Index at the final training epoch.  
A decaying curve from right (output) to left (input) visually demonstrates vanishing gradients.

In [ ]:
# Cell 18: Run Experiment 2
# ReLU runs are automatically reused (AUTO_RESUME skips existing run_ids)
# Only Sigmoid runs are new.

runner.run_experiment_2()
print(f"\nExperiment 2 complete.")

In [ ]:
#  Cell 19: Figure 4A - Gradient Norm vs Layer Depth (PRIMARY FIGURE)
plot_gradient_norm_by_layer(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"],
)


In [ ]:
#  Cell 20: Figure 4B - Gradient Norm vs Epoch (SUPPLEMENTAL)
plot_gradient_norm_over_epochs(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"],
)

In [ ]:
# Cell 20B: Figure 3 — Test Accuracy Bar Chart (ALL 6 configs)
plot_test_accuracy_bar(
    summary_path = CONFIG["summary_path"],
    figures_dir  = CONFIG["figures_dir"],
)


### Experiment 2 Analysis

**Gradient Norm Analysis (Fig 4A & 4B):**

**The Vanishing Gradient Pathology:** While shallow Sigmoid models are competitive (2L Sigmoid = 88.59%), deep Sigmoid networks degrade severely (8L Sigmoid = 86.74%). Figure 4B provides the mechanistic "smoking gun" for this failure: at epoch 1, the Layer 1 gradient for the 8L Sigmoid model collapses to ~10⁻⁹. This represents a 9-order-of-magnitude deficit compared to ReLU.

**Convergence Penalty:** Because the early layers are starved of gradients, the 8L Sigmoid model requires ~12 epochs just to converge, compared to 2-3 epochs for all ReLU models. While the Sigmoid gradients eventually recover after epoch 5, the model never fully closes the accuracy gap.

---
## Section 6: Experiment 3 — BatchNorm Recovery

**Research Question (RQ4):** Can BatchNorm restore trainability to a deep Sigmoid network?

**Setup:**
- Models: 8L Sigmoid (no BN) vs 8L Sigmoid + BatchNorm
- 8L Sigmoid (no BN) runs are reused from Experiment 2 (AUTO_RESUME)

**BatchNorm Mechanism:**

BatchNorm normalises each layer's pre-activations:
```
ẑ = (z − μ_B) / √(σ²_B + ε)     [normalise to zero mean, unit variance]
y = γẑ + β                          [learnable scale and shift]
```

For Sigmoid networks specifically:
- By keeping pre-activations near zero, BN ensures inputs arrive in Sigmoid's **linear region** (near x=0), where σ'(0) = 0.25 — the **maximum** of the sigmoid derivative
- Without BN, deep Sigmoid networks saturate at ±∞ where σ'(x) → 0
- BN breaks the exponential gradient decay by resetting the pre-activation distribution at each layer

In [ ]:
# Cell 23: Run Experiment 3
# 8L Sigmoid (no BN) is reused from Experiment 2.
# Only 8L Sigmoid + BN is new (3 seeds).

runner.run_experiment_3()
print(f"\nExperiment 3 complete.")

In [ ]:
# Cell 24: Figure 6 - BatchNorm Recovery
plot_batchnorm_recovery(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"],
)
# Panel A: 8L Sigmoid+BN should show substantially higher val_acc than 8L Sigmoid
# Panel B: Gradient attenuation ratio for BN model should be much closer to 1.0
#          than the no-BN Sigmoid model

### Experiment 3 Analysis

**BatchNorm Recovery (Fig 6):**

**Convergence vs. Accuracy:** BatchNorm acts as a powerful optimization rescue. By normalizing pre-activations, it prevents early Sigmoid saturation and completely restores convergence speed (12 epochs down to 2-3 epochs).

**The Residual Gap:** However, BatchNorm only *partially* recovers the accuracy penalty (86.74% → 88.12%). It falls short of the ReLU baseline (88.72%). This indicates that the performance degradation of deep Sigmoid networks is caused by both optimization difficulties and activation-function limitations. BatchNorm addresses the optimization component but cannot eliminate the bounded derivative of the Sigmoid activation.

---
## Section 7: Summary Table

All results reported as **mean ± std** across 3 seeds.

| Depth | Activation | BatchNorm | Mean Test Acc | Std Dev | Convergence Epoch |
|-------|------------|-----------|---------------|---------|-------------------|
| 2L    | ReLU       | Off       | 0.8879        | 0.0033  | 1.3               |
| 4L    | ReLU       | Off       | 0.8879        | 0.0027  | 1.3               |
| 8L    | ReLU       | Off       | 0.8872        | 0.0022  | 2.3               |
| 2L    | Sigmoid    | Off       | 0.8859        | 0.0039  | 3.3               |
| 4L    | Sigmoid    | Off       | 0.8810        | 0.0027  | 3.7               |
| 8L    | Sigmoid    | Off       | 0.8674        | 0.0020  | 12.0              |
| 8L    | Sigmoid    | On        | 0.8812        | 0.0017  | 2.3               |


In [ ]:
# Cell 27: Aggregate Summary Table
agg = compute_aggregate_summary(CONFIG["summary_path"])

if agg.empty:
    print("No results yet — run experiments first.")
else:
    def fmt_mean_std(m, s):
        if pd.isna(s) or s == 0:
            return f"{m:.4f}"
        return f"{m:.4f} ± {s:.4f}"

    display_rows = []
    for _, row in agg.iterrows():
        display_rows.append({
            "Depth"      : int(row["depth"]),
            "Width"      : int(row["width"]),
            "Activation" : row["activation"],
            "BatchNorm"  : "Yes" if row["batchnorm"] else "No",
            "Parameters" : f"{int(row['parameter_count']):,}",
            "Test Acc"   : fmt_mean_std(row["mean_test_acc"], row["std_test_acc"]),
            "Conv.Epoch" : fmt_mean_std(row["mean_convergence_epoch"], row["std_convergence_epoch"]),
            "Grad Ratio" : fmt_mean_std(row["mean_gradient_ratio"], row["std_gradient_ratio"]),
            "Seeds"      : int(row["n_seeds"]),
        })

    table_df = pd.DataFrame(display_rows)
    print("=" * 100)
    print("SUMMARY TABLE: All Results (mean ± std across seeds)")
    print("=" * 100)
    print(table_df.to_string(index=False))
    print("\nGrad Ratio = ‖∇W_layer1‖ / ‖∇W_lastHidden‖  (higher → more vanishing gradient)")

In [ ]:
# Cell 28: Figure 5 - Activation Comparison Heatmap
plot_activation_heatmap(
    summary_path = CONFIG["summary_path"],
    figures_dir  = CONFIG["figures_dir"],
)


---
## Section 8: Discussion

### Observation: Overfitting Dynamics

All ReLU models reach minimum validation loss around epochs 8–12 and then begin to overfit.

Validation accuracy remains relatively stable, but validation loss increases steadily.

This indicates increasing prediction confidence without corresponding generalization gains.

The deeper 8-layer model appears to overfit slightly less than the shallower models. This suggests that deeper, narrower MLPs may provide stronger implicit regularization under Adam.

### Unexpected Result: Depth and ReLU

Under a controlled parameter budget, increasing depth from 2 to 8 hidden layers did not produce a statistically significant improvement in test accuracy for ReLU networks.

- 2L ReLU: 88.79%
- 4L ReLU: 88.79%
- 8L ReLU: 88.72%

This suggests that trainability rather than depth alone determines practical performance on Fashion-MNIST. The representational capacity provided by a shallow ReLU MLP is already sufficient to model the task effectively.

### The Vanishing Gradient Pathology

While shallow Sigmoid models are competitive (2L Sigmoid = 88.59%), deep Sigmoid networks degrade severely (8L Sigmoid = 86.74%). Figure 4B provides the mechanistic "smoking gun" for this failure: at epoch 1, the Layer 1 gradient for the 8L Sigmoid model collapses to ~10⁻⁹. This represents a 9-order-of-magnitude deficit compared to ReLU, explaining the severe convergence penalty (12 epochs vs 2-3 epochs).

### BatchNorm Recovery

BatchNorm acts as a powerful optimization rescue, completely restoring convergence speed (12 epochs down to 2-3 epochs) by preventing early Sigmoid saturation. However, it only *partially* recovers the accuracy penalty (86.74% → 88.12%), falling short of the ReLU baseline (88.72%). This indicates that the performance degradation of deep Sigmoid networks is caused by both optimization difficulties and activation-function limitations. BatchNorm addresses the optimization component but cannot eliminate the bounded derivative of the Sigmoid activation.

---
## Section 9: Conclusion

### Limitations

- Only Fashion-MNIST evaluated
- Only Adam optimizer used
- Only 3 seeds
- No early stopping
- No regularization
- No CNN baseline

### Future Work

- CIFAR-10 evaluation
- SGD comparison
- Residual connections
- Early stopping
- Dropout and weight decay
- Deeper architectures (>8 layers)

### Direct Answers to Research Questions

**RQ1: Does increasing depth improve performance under a fixed parameter budget?**  
**No.** Under strict capacity control (~500k parameters), increasing depth provides zero performance benefit for Fashion-MNIST. ReLU accuracy remained static (88.79% at 2L vs 88.72% at 8L), while Sigmoid accuracy systematically degraded (88.59% at 2L vs 86.74% at 8L).

**RQ2: How does depth affect gradient flow?**  
Depth introduces severe gradient attenuation for saturating activations. In 8L Sigmoid, gradients at the input layer collapsed to 10⁻⁹ at initialization. Non-saturating activations (ReLU) bypass this decay, preserving stable gradient flow regardless of depth.

**RQ3: Can architectural interventions rescue deep network trainability?**  
**Partially.** Batch Normalization rescued the trainability of the 8L Sigmoid model (accelerating convergence from 12 epochs to 2-3 epochs), but the final accuracy (88.12%) remained structurally bottlenecked below the ReLU ceiling (88.72%).

---
## Section 10: Final Execution Summary

This cell aggregates the final metrics, verifies that all outputs were generated successfully, and provides a quick verification printout of the entire pipeline.

In [ ]:
# Cell 31: Final Execution Verification & Artifact Export
import os
import shutil
import sys
from pathlib import Path
import pandas as pd

log_lines = []
def log_print(msg):
    print(msg)
    log_lines.append(str(msg))

log_print("=" * 80)
log_print("FINAL PIPELINE VERIFICATION")
log_print("=" * 80)

# 1. Verify Dataset
log_print("\n[1] DATASET VERIFICATION")
log_print(f"Train size: {len(train_loader.dataset):,}")
log_print(f"Val size  : {len(val_loader.dataset):,}")
log_print(f"Test size : {len(test_loader.dataset):,}")

# 2. Verify Figures
log_print("\n[2] FIGURES GENERATED")
fig_dir = Path(CONFIG['figures_dir'])
expected_figs = [
    "dataset_samples.png",
    "fig1_val_accuracy.png",
    "fig2_val_loss.png",
    "fig3_test_accuracy_bar.png",
    "fig4a_gradient_by_layer.png",
    "fig4b_gradient_over_epochs.png",
    "fig5_activation_heatmap.png",
    "fig6_batchnorm_recovery.png"
]
for f in expected_figs:
    f_path = fig_dir / f
    status = "✅ Found" if f_path.exists() else "❌ Missing"
    log_print(f"  {status} : {f}")

# 3. Verify Results CSV
log_print("\n[3] EXPERIMENT RESULTS")
summary_path = Path(CONFIG['summary_path'])
if summary_path.exists():
    df = pd.read_csv(summary_path)
    log_print(f"  ✅ Found summary CSV with {len(df)} total runs recorded.")
    log_print("\n  High-Level Test Accuracies (Averaged across seeds):")
    agg = compute_aggregate_summary(summary_path)
    for _, row in agg.iterrows():
        bn_str = "+ BN" if row['batchnorm'] else ""
        log_print(f"    - {int(row['depth'])}L {row['activation']} {bn_str:>4} : "
              f"{row['mean_test_acc']:.4f} ± {row['std_test_acc']:.4f}")
else:
    log_print("  ❌ Missing summary CSV!")

log_print("\n" + "=" * 80)
log_print("PIPELINE COMPLETE. Notebook successfully executed top-to-bottom.")
log_print("=" * 80)

# 4. Save Log and Export Artifacts
results_dir = Path(CONFIG['results_path']).parent
results_dir.mkdir(exist_ok=True)
log_path = Path(WORKSPACE) / 'final_verification_log.txt'
with open(log_path, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))

zip_path = f"{CONFIG['exports_dir']}/fashion_mnist_final_results.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', WORKSPACE)

print(f"\n✅ All results packaged into {zip_path}")
print(f"✅ Text verification log saved to {log_path}")

# Auto-download if on Colab
if 'google.colab' in sys.modules:
    try:
        from google.colab import files
        print("Triggering automatic download of verification log and results zip...")
        files.download(str(log_path))
        files.download(zip_path)
    except Exception as e:
        print(f"Failed to auto-download: {e}")
